# Fareez n=40 RAG Ablation Analysis (Table 2c)

This notebook is the n=40 statistical replacement for the n=6 institutional ablations (paper Table 2b). It reads the consolidated metrics produced by `rag_models/evaluation/compute_ablation_metrics.py` and reproduces every numeric claim that goes into Table 2c and the revised RAG-Architecture paragraph of the manuscript.

**Models** (4 — same set as the clinician evaluation): GPT-OSS-120B, GPT-OSS-20B, Qwen3-32B, MedGemma-4B.

**Cells** (one row per cell):
- `production` — the existing RAG-on (k=2, all-MiniLM-L6-v2) Fareez run, loaded from `rag_models/results/fareez/fareez_automated_metrics.csv` for direct comparability.
- `norag` — schema retrieval disabled.
- `k1`, `k3`, `k5` — top-k sweep.
- `embed_clinicalbert`, `embed_pubmedbert` — embedding-substitution.
- `temp_0.1`, `temp_0.5`, `temp_0.7` — temperature sweep on top-2 models.
- `prompt_minimal`, `prompt_hallucination_guarded` — prompt-variant ablation on top-2 models.

**Reference** for all metrics: source transcript (transcript-as-reference). This matches the methodology used for Table 3 since Fareez has no gold-standard SOAP references.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

HERE = Path('.').resolve()
RAG_ROOT = HERE.parent.parent.parent

abl = pd.read_csv(HERE / 'fareez_ablation_metrics.csv')
main = pd.read_csv(RAG_ROOT / 'results' / 'fareez' / 'fareez_automated_metrics.csv')

# Add 'cell' = 'production' to the main run for unified analysis
main = main.copy()
main['cell'] = 'production'

# Common columns
common = ['cell','model','conversation','bleu','rouge_l','sbert_coherence','bert_f1',
          'summary_chars','summary_tokens']
df = pd.concat([abl[common], main[common]], ignore_index=True)
print(f'Total rows: {len(df)}')
print(f'Cells:  {sorted(df.cell.unique())}')
print(f'Models: {sorted(df.model.unique())}')
print(f'N per cell:  {df.groupby("cell").conversation.count().to_dict()}')

## Table 2c — per-(cell, model) means

This is the table that goes into the paper. Rows = (cell, model). Columns = the four most-comparable metrics.

In [ ]:
metric_cols = ['bleu','rouge_l','sbert_coherence','bert_f1']
table = df.groupby(['cell','model'])[metric_cols].mean().round(4)
table

## Delta vs. production (the no-RAG / ksweep / embedding effects)

For each non-production cell, compute the per-model delta against `production` for each metric. Positive = production better than the cell.

In [ ]:
prod = table.loc['production']
rows = []
for cell in sorted(c for c in table.index.get_level_values('cell').unique() if c != 'production'):
    for model in sorted(table.loc[cell].index):
        if model not in prod.index:
            continue
        row = {'cell': cell, 'model': model}
        for m in metric_cols:
            row[f'delta_{m}'] = round(prod.loc[model, m] - table.loc[(cell, model), m], 4)
        rows.append(row)
deltas = pd.DataFrame(rows)
deltas

## Statistical test: Wilcoxon signed-rank (production vs. each ablation cell), per model

Pairs each conversation with itself (same conversation, same model, two configs). Two-sided.

In [ ]:
from scipy import stats
rows = []
for cell in sorted(c for c in df.cell.unique() if c != 'production'):
    for model in sorted(df.model.unique()):
        for m in metric_cols:
            a = df[(df.cell == 'production') & (df.model == model)].set_index('conversation')[m]
            b = df[(df.cell == cell) & (df.model == model)].set_index('conversation')[m]
            common_ids = a.index.intersection(b.index)
            if len(common_ids) < 5:
                continue
            try:
                stat, p = stats.wilcoxon(a.loc[common_ids].values, b.loc[common_ids].values)
            except ValueError:
                stat, p = (np.nan, np.nan)
            rows.append({'cell': cell, 'model': model, 'metric': m,
                         'n': len(common_ids), 'wilcoxon_stat': stat, 'p': p,
                         'mean_diff': round(a.loc[common_ids].mean() - b.loc[common_ids].mean(), 4)})
tests = pd.DataFrame(rows)
tests['p'] = tests['p'].apply(lambda x: f'{x:.4f}' if pd.notna(x) else '')
tests

## Bootstrap 95% CI for the no-RAG → production deltas

Reviewer #1's critique of the n=6 ablation is that single anomalous cases could flip conclusions. With n=40 we can compute proper bootstrap CIs.

In [ ]:
rng = np.random.default_rng(42)
boot_rows = []
for cell in sorted(c for c in df.cell.unique() if c != 'production'):
    for model in sorted(df.model.unique()):
        for m in metric_cols:
            a = df[(df.cell == 'production') & (df.model == model)].set_index('conversation')[m]
            b = df[(df.cell == cell) & (df.model == model)].set_index('conversation')[m]
            common_ids = list(a.index.intersection(b.index))
            if len(common_ids) < 5:
                continue
            diffs = a.loc[common_ids].values - b.loc[common_ids].values
            B = 5000
            idx = rng.integers(0, len(diffs), size=(B, len(diffs)))
            boot_means = diffs[idx].mean(axis=1)
            lo, hi = np.percentile(boot_means, [2.5, 97.5])
            boot_rows.append({
                'cell': cell, 'model': model, 'metric': m,
                'n': len(diffs),
                'mean_delta': round(diffs.mean(), 4),
                'ci_lo': round(lo, 4),
                'ci_hi': round(hi, 4),
                'ci_excludes_zero': not (lo <= 0 <= hi),
            })
boot = pd.DataFrame(boot_rows)
boot

In [ ]:
# Save the three publication artefacts
table.to_csv(HERE / 'table2c_means.csv')
deltas.to_csv(HERE / 'table2c_deltas.csv', index=False)
boot.to_csv(HERE / 'table2c_bootstrap_ci.csv', index=False)
tests.to_csv(HERE / 'table2c_wilcoxon.csv', index=False)
print('Saved:')
for f in ['table2c_means.csv','table2c_deltas.csv','table2c_bootstrap_ci.csv','table2c_wilcoxon.csv']:
    print(' -', HERE / f)

## Temperature & prompt sweeps (top-2 models only)

In [ ]:
temp_cells = [c for c in df.cell.unique() if c.startswith('temp_')] + ['production']
if any(c.startswith('temp_') for c in df.cell.unique()):
    print('Temperature sweep:')
    print(df[(df.cell.isin(temp_cells)) & (df.model.isin(['gpt-oss-120b','gpt-oss-20b']))]
          .groupby(['cell','model'])[metric_cols].mean().round(4))

In [ ]:
prompt_cells = [c for c in df.cell.unique() if c.startswith('prompt_')] + ['production']
if any(c.startswith('prompt_') for c in df.cell.unique()):
    print('Prompt-variant sweep:')
    print(df[(df.cell.isin(prompt_cells)) & (df.model.isin(['gpt-oss-120b','gpt-oss-20b']))]
          .groupby(['cell','model'])[metric_cols].mean().round(4))